# 1. Applying ML Model to Application Data

In [77]:
import pandas as pd
import numpy as np
import joblib

loaded_pipeline = joblib.load('credit_scoring_model.pkl')
df = pd.read_csv('New_MSME_credit_application.csv')

# Preprocessing
df['Loan_to_Value'] = np.where(
    df['Collateral_Value'] > 0,
    df['Loan_Ceiling'] / df['Collateral_Value'],
    0
)
tenor_yrs = 3
interest_rate = 0.1
df['Annual_Loan'] = df['Loan_Ceiling'] / tenor_yrs + df['Loan_Ceiling'] * interest_rate
df['Debt_to_Asset_Ratio'] = df['Annual_Loan'] / df['Asset']
df['Debt_to_Income_Ratio'] = df['Annual_Loan'] / df['Revenue']
df['Debt_Service_Coverage_Ratio'] = df['Profit'] / df['Annual_Loan']
df

# Apply ML to calculate PD, Credit Score, and ECL
probability = loaded_pipeline.predict_proba(df)[:, 1]
probability_clipped = np.clip(probability, 0.0001, 0.9999)

df['Probability_of_Default'] = probability_clipped
df['Odds'] = (1 - df['Probability_of_Default']) / df['Probability_of_Default']

base_score = 600
base_odds = df['Odds'].median()
pdo = 100 / (np.log(df['Odds']).std())
factor = pdo / np.log(2)
offset = base_score - (factor * np.log(base_odds))

df['Credit_Score'] = offset + (factor * np.log(df['Odds']))
df['Credit_Score'] = np.clip(np.round(df['Credit_Score']), 300, 850).astype(int)

df['log_DSCR'] = np.log(df['Debt_Service_Coverage_Ratio'])
DSCR_min = df['log_DSCR'].min()
DSCR_max = df['log_DSCR'].max()
df['DSCR_norm'] = (df[['log_DSCR']] - DSCR_min)/(DSCR_max - DSCR_min)
df['EAD_Factor'] = 0.9 - df['DSCR_norm']*(0.9 - 0.4)
df['Exposure_At_Default'] = df['EAD_Factor'] * df['Loan_Ceiling']

df['Loss_Given_Default'] = np.where(df['Business_Scale'].isin(['Micro', 'Small']), 0.8, 0.4)

df['Expected_Credit_Loss'] = df['Probability_of_Default'] * df['Exposure_At_Default'] * df['Loss_Given_Default']

# Calculating Risk Matrix
CS_conditions = [
    df['Credit_Score'] <= 580,
    (df['Credit_Score'] > 580) & (df['Credit_Score'] <= 670),
    (df['Credit_Score'] > 670) & (df['Credit_Score'] <= 740),
    (df['Credit_Score'] > 740) & (df['Credit_Score'] <= 800),
    df['Credit_Score'] > 800
]
Likelihood_vals = ['Very Likely', 'Likely', 'Possible', 'Unlikely', 'Very Unlikely']
df['Likelihood'] = np.select(CS_conditions, Likelihood_vals, default=Likelihood_vals[-1])

df['log_ECL'] = np.log(df['Expected_Credit_Loss'])
ECL_min = df['log_ECL'].min()
ECL_max = df['log_ECL'].max()
ECL_step = (ECL_max - ECL_min)/5
ECL_conditions = [
    df['log_ECL'] <= ECL_min + ECL_step,
    (df['log_ECL'] > ECL_min + ECL_step) & (df['log_ECL'] <= ECL_min + 2*ECL_step),
    (df['log_ECL'] > ECL_min + 2*ECL_step) & (df['log_ECL'] <= ECL_min + 3*ECL_step),
    (df['log_ECL'] > ECL_min + 3*ECL_step) & (df['log_ECL'] <= ECL_min + 4*ECL_step),
    df['log_ECL'] > ECL_min + 4*ECL_step
]
Severity_vals = ['Negligible', 'Minor', 'Moderate', 'Significant', 'Severe']
df['Severity'] = np.select(ECL_conditions, Severity_vals, default=Severity_vals[-1])

Credit_conditions = [
    (df['Severity'] == 'Negligible') |
    ((df['Severity'] == 'Minor') & (df['Likelihood'] != 'Very Likely')) |
    ((df['Severity'] == 'Moderate') & ((df['Likelihood'] == 'Very Unlikely') | (df['Likelihood'] == 'Unlikely'))),

    ((df['Severity'] == 'Minor') & (df['Likelihood'] == 'Very Likely')) |
    ((df['Severity'] == 'Moderate') & (df['Likelihood'] != 'Very Unlikely') & (df['Likelihood'] != 'Unlikely')) |
    ((df['Severity'] == 'Significant') & (df['Likelihood'] != 'Very Likely')) |
    ((df['Severity'] == 'Severe') & (df['Likelihood'] != 'Likely') & (df['Likelihood'] != 'Very Likely')),

    ((df['Severity'] == 'Significant') & (df['Likelihood'] == 'Very Likely')) |
    ((df['Severity'] == 'Severe') & ((df['Likelihood'] == 'Likely') | (df['Likelihood'] == 'Very Likely')))
]
Approval_vals = ['Accepted', 'Referred', 'Denied']
df['Approval_Status'] = np.select(Credit_conditions, Approval_vals, default=Approval_vals[-1])

df = df.drop(columns=['log_DSCR', 'DSCR_norm', 'EAD_Factor', 'log_ECL'])
df.to_csv('New_MSME_credit_reviewed.csv', index=False)
df

,Customer_ID,Owner_Age,Marital_Status,Gender,Business_Scale,Business_Sector,Business_Age,Asset,Revenue,Profit,...,Debt_Service_Coverage_Ratio,Probability_of_Default,Odds,Credit_Score,Exposure_At_Default,Loss_Given_Default,Expected_Credit_Loss,Likelihood,Severity,Approval_Status
0,100001,30,Divorced,Male,1. Micro,5. Others,4,5.390496e+08,2.619361e+08,6.994558e+07,...,2.131925,0.250329,2.994748,345,5.551490e+07,0.4,5.558788e+06,Very Likely,Significant,Denied
1,100002,53,Married,Female,2. Small,3. Service,18,2.268739e+09,4.184577e+09,8.141834e+08,...,4.041636,0.092511,9.809507,628,3.197299e+08,0.4,1.183143e+07,Likely,Significant,Referred
2,100003,53,Married,Female,1. Micro,4. Trade,18,9.602908e+08,1.404631e+09,2.918579e+08,...,11.030460,0.089805,10.135187,636,3.763680e+07,0.4,1.351995e+06,Likely,Moderate,Referred
3,100004,46,Married,Female,2. Small,4. Trade,10,2.583360e+09,1.005175e+10,1.082245e+09,...,11.915399,0.064990,14.386974,719,1.280470e+08,0.4,3.328712e+06,Possible,Moderate,Referred
4,100005,40,Married,Male,1. Micro,3. Service,19,2.851716e+08,1.583290e+09,4.243240e+08,...,28.934989,0.089805,10.135187,636,1.853967e+07,0.4,6.659850e+05,Likely,Moderate,Referred
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49961,149962,57,Married,Male,3. Medium,1. Culinary,24,7.615106e+09,4.551538e+10,4.499012e+09,...,2.365585,0.132485,6.548002,532,3.185653e+09,0.4,1.688210e+08,Very Likely,Severe,Denied
49962,149963,51,Married,Male,1. Micro,3. Service,9,6.223619e+08,9.181564e+08,2.609329e+08,...,111.367829,0.085917,10.639162,647,2.444036e+06,0.4,8.399355e+04,Likely,Minor,Accepted
49963,149964,27,Married,Male,3. Medium,4. Trade,6,6.774325e+09,4.344039e+10,5.362975e+09,...,3.796054,0.137828,6.255404,521,2.256818e+09,0.4,1.244214e+08,Very Likely,Severe,Denied
49964,149965,34,Married,Female,3. Medium,4. Trade,19,6.327873e+09,4.016517e+10,4.397124e+09,...,2.392988,0.132485,6.548002,532,3.074384e+09,0.4,1.629244e+08,Very Likely,Severe,Denied


# 2. Generate Controlled-Random Credit Status

In [78]:
np.random.seed(123)
df = pd.read_csv('New_MSME_credit_reviewed.csv')

df = df[df['Approval_Status'] != 'Denied']

df['Prob_Non_Performing'] = np.random.uniform(0.009, 0.011)

scale_micro = df['Business_Scale'] == '1. Micro'
df.loc[scale_micro, 'Prob_Non_Performing'] += np.random.uniform(0.03,0.05)

sektor_culinary = df['Business_Sector'] == '1. Culinary'
df.loc[sektor_culinary, 'Prob_Non_Performing'] += np.random.uniform(0.04, 0.06)

df['Prob_Non_Performing'] += 1/df['Business_Age']/np.random.uniform(9,11)
df['Prob_Non_Performing'] += df['Loan_Ceiling'] / df['Profit'] / np.random.uniform(20,30)
df['Prob_Non_Performing'] = np.clip(df['Prob_Non_Performing'],0,1)

df['Credit_Status'] = np.random.binomial(n=1, p=df['Prob_Non_Performing'])

df[['Customer_ID', 'Credit_Status']].to_csv('New_MSME_credit_actual.csv', index=False)
df

,Customer_ID,Owner_Age,Marital_Status,Gender,Business_Scale,Business_Sector,Business_Age,Asset,Revenue,Profit,...,Odds,Credit_Score,Exposure_At_Default,Loss_Given_Default,Expected_Credit_Loss,Likelihood,Severity,Approval_Status,Prob_Non_Performing,Credit_Status
1,100002,53,Married,Female,2. Small,3. Service,18,2.268739e+09,4.184577e+09,8.141834e+08,...,9.809507,628,3.197299e+08,0.4,1.183143e+07,Likely,Significant,Referred,0.036888,0
2,100003,53,Married,Female,1. Micro,4. Trade,18,9.602908e+08,1.404631e+09,2.918579e+08,...,10.135187,636,3.763680e+07,0.4,1.351995e+06,Likely,Moderate,Referred,0.059308,1
3,100004,46,Married,Female,2. Small,4. Trade,10,2.583360e+09,1.005175e+10,1.082245e+09,...,14.386974,719,1.280470e+08,0.4,3.328712e+06,Possible,Moderate,Referred,0.027413,0
4,100005,40,Married,Male,1. Micro,3. Service,19,2.851716e+08,1.583290e+09,4.243240e+08,...,10.135187,636,1.853967e+07,0.4,6.659850e+05,Likely,Moderate,Referred,0.054258,0
6,100007,57,Married,Male,1. Micro,1. Culinary,8,2.987704e+08,5.827292e+08,1.295682e+08,...,6.515177,531,3.374389e+07,0.4,1.796040e+06,Very Likely,Moderate,Referred,0.117506,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49957,149958,36,Married,Male,1. Micro,2. Manufacture,10,9.286119e+07,1.129775e+09,2.804495e+08,...,7.039973,549,4.009458e+07,0.4,1.994762e+06,Very Likely,Moderate,Referred,0.064453,0
49958,149959,42,Married,Male,2. Small,4. Trade,20,3.629602e+09,1.319835e+10,2.087353e+09,...,21.192431,812,1.061972e+08,0.4,1.914116e+06,Very Unlikely,Moderate,Accepted,0.018698,0
49960,149961,48,Divorced,Male,3. Medium,4. Trade,24,9.399016e+09,4.439228e+10,6.476607e+09,...,19.852142,796,3.692140e+08,0.4,7.082514e+06,Unlikely,Significant,Referred,0.018230,0
49962,149963,51,Married,Male,1. Micro,3. Service,9,6.223619e+08,9.181564e+08,2.609329e+08,...,10.639162,647,2.444036e+06,0.4,8.399355e+04,Likely,Minor,Accepted,0.057876,1
